In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%pip install brian2 brian2hears librosa numpy scipy matplotlib setuptools

import logging
logging.getLogger('brian2').setLevel(logging.ERROR)
import brian2
brian2.prefs.codegen.target = 'numpy'
from scipy import signal
import numpy as np
import os
import glob
import tensorflow as tf

def process_eeg_file(npz_filename, mode='cnn'):
    data = np.load(npz_filename)
    eeg_data = data['eeg']
    fs = int(data['fs'])

    attended_wav = str(data['stimulus_attended'])
    unattended_wav = str(data['stimulus_unattended'])

    if mode == 'linear':
        target_sr = 20
        lowcut = 1.0
        highcut = 9.0
    elif mode == 'cnn':
        target_sr = 64
        lowcut = 1.0
        highcut = 32.0
    else:
        raise ValueError("Kies 'linear' of 'cnn' als mode.")

    sos = signal.butter(N=4, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')
    eeg_filtered = signal.sosfiltfilt(sos, eeg_data, axis=0)

    from math import gcd
    g = gcd(fs, target_sr)
    eeg_downsampled = signal.resample_poly(eeg_filtered, target_sr // g, fs // g, axis=0)

    return eeg_downsampled, attended_wav, unattended_wav

In [ ]:
def batch_equalizer(eeg, env_1, env_2, labels):
    return (np.concatenate([eeg,eeg], axis=0),
            np.concatenate([env_1, env_2], axis=0),
            np.concatenate([env_2, env_1], axis=0)), \
           np.concatenate([labels, (labels+1)%2], axis=0)

class DataGenerator:
    def __init__(self, files, audio_dir, time_window=640):
        self.files = files
        self.audio_dir = audio_dir
        self.time_window = time_window

        alle_audio = glob.glob(os.path.join(self.audio_dir, "**", "*.npy"), recursive=True)
        self.audio_dict = {os.path.basename(f).lower(): f for f in alle_audio}
        print(f" {len(self.audio_dict)} audiobestanden gevonden in de map!")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, recording_index):
        eeg_pad = self.files[recording_index]
        eeg_processed, att_naam, unatt_naam = process_eeg_file(eeg_pad, mode='cnn')

        att_base = att_naam.replace('.wav', '').replace('.npy', '').strip()
        unatt_base = unatt_naam.replace('.wav', '').replace('.npy', '').strip()

        # voegt _cnn.npy toe
        zoek_att = f"{att_base.lower()}_cnn.npy"
        zoek_unatt = f"{unatt_base.lower()}_cnn.npy"

        att_audio_pad = self.audio_dict.get(zoek_att)
        unatt_audio_pad = self.audio_dict.get(zoek_unatt)

        if not att_audio_pad or not unatt_audio_pad:
            raise FileNotFoundError(f"Audio '{zoek_att}' of '{zoek_unatt}' niet gevonden!")

        env1 = np.load(att_audio_pad)
        env2 = np.load(unatt_audio_pad)

        min_len = min(eeg_processed.shape[0], env1.shape[0], env2.shape[0])
        eeg_processed = eeg_processed[:min_len]
        env1 = env1[:min_len]
        env2 = env2[:min_len]

        num_windows = min_len // self.time_window
        eeg_windows = np.array(np.split(eeg_processed[:num_windows * self.time_window], num_windows))
        env1_windows = np.array(np.split(env1[:num_windows * self.time_window], num_windows))
        env2_windows = np.array(np.split(env2[:num_windows * self.time_window], num_windows))

        env1_windows = np.expand_dims(env1_windows, axis=-1)
        env2_windows = np.expand_dims(env2_windows, axis=-1)

        labels = np.ones((num_windows, 1))

        (eeg_bal, env1_bal, env2_bal), labels_bal = batch_equalizer(eeg_windows, env1_windows, env2_windows, labels)

        return (eeg_bal, env1_bal, env2_bal), labels_bal

    def __call__(self):
        for idx in range(self.__len__()):
            try:
                yield self.__getitem__(idx)
            except Exception as e:
                print(f" Fout bij inladen trial: {e}")
                continue

            if idx == self.__len__() - 1:
                self.on_epoch_end()

    def on_epoch_end(self):
        np.random.shuffle(self.files)

In [ ]:
import random
pad_naar_eeg = "/content/drive/MyDrive/p&d/fase_2/data/64 Channel Biosemi unprocessed data - train/*/*/*.npz"
pad_naar_audio = "/content/drive/MyDrive/p&d/fase_2/data/preprocessed/audio/cnn/"

alle_eeg_bestanden = glob.glob(pad_naar_eeg)
print(f"Totaal aantal EEG trials gevonden: {len(alle_eeg_bestanden)}")

random.seed(42) 
random.shuffle(alle_eeg_bestanden)

split_index = int(len(alle_eeg_bestanden) * 0.8)
train_files = alle_eeg_bestanden[:split_index]
val_files = alle_eeg_bestanden[split_index:]

print(f"Trainingsset: {len(train_files)} trials")
print(f"Validatieset: {len(val_files)} trials")


def maak_tf_dataset(files, audio_dir):
    generator = DataGenerator(files=files, audio_dir=audio_dir, time_window=640)
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            (tf.TensorSpec(shape=(None, 640, 64), dtype=tf.float32), 
             tf.TensorSpec(shape=(None, 640, 1), dtype=tf.float32),
             tf.TensorSpec(shape=(None, 640, 1), dtype=tf.float32)),
            tf.TensorSpec(shape=(None, 1), dtype=tf.float32)
        )
    )
    return dataset

train_dataset = maak_tf_dataset(train_files, pad_naar_audio)
val_dataset = maak_tf_dataset(val_files, pad_naar_audio)

time_window = 640

eeg_input = tf.keras.layers.Input(shape=[time_window, 64], name="EEG_Input")
env1_input = tf.keras.layers.Input(shape=[time_window, 1], name="Env1_Input")
env2_input = tf.keras.layers.Input(shape=[time_window, 1], name="Env2_Input")

eeg_conv = tf.keras.layers.Conv1D(filters=8, kernel_size=16, activation='relu', name="EEG_Conv")(eeg_input) 

env_conv_layer = tf.keras.layers.Conv1D(filters=1, kernel_size=16, activation='relu', name="Env_Conv") 
env1_processed = env_conv_layer(env1_input) 
env2_processed = env_conv_layer(env2_input)

cos1 = tf.keras.layers.Dot(axes=1, normalize=True, name="Cosine_Env1")([eeg_conv, env1_processed]) 
cos2 = tf.keras.layers.Dot(axes=1, normalize=True, name="Cosine_Env2")([eeg_conv, env2_processed])

concat = tf.keras.layers.Concatenate()([cos1, cos2])
flat = tf.keras.layers.Flatten()(concat)
out1 = tf.keras.layers.Dense(1, activation="sigmoid", name="Decision_Neuron")(flat) 
out = tf.keras.layers.Reshape([1], name="output_name")(out1)

model = tf.keras.Model(inputs=[eeg_input, env1_input, env2_input], outputs=[out])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

geschiedenis = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=80
)